# AutoInsight — Vehicle Parts Segmentation (Kaggle GPU)

**Goal:** train a segmentation model to identify vehicle parts for AutoInsight's damage-to-part fusion pipeline.

**Model:** YOLO26m-seg (medium segmentation model; higher capacity than the original YOLO26s-seg). If this checkpoint is unavailable in the installed Ultralytics version, the setup cell tries YOLO11m-seg. This is a strong starting point, not a guarantee of maximum accuracy.

**Accuracy priorities**
- Use the correct attached dataset and verify class IDs before training.
- Keep the train/validation/test splits separate.
- Merge sparse left/right labels only if your backend's class schema expects merged classes.
- Use validation results for decisions; use the test set only for final reporting.
- Real service-centre photos with consistent polygon labels are likely to improve real-world performance more than blindly increasing epochs.

This notebook is migrated for Kaggle: no Google Drive mounting; datasets are read from `/kaggle/input`; generated data, runs and exports go to `/kaggle/working`.


In [4]:
%pip install -q -U ultralytics opencv-python-headless pyyaml

import os, json, time, shutil, zipfile, urllib.request, random
from pathlib import Path
from collections import Counter
import yaml
import torch
import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle Notebook Settings."


Note: you may need to restart the kernel to use updated packages.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.184
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
# Kaggle does not use Google Drive mounting. This cell intentionally replaces the old Colab mount.
print("Kaggle input root exists:", os.path.isdir("/kaggle/input"))


Kaggle input root exists: True


## Config

In [13]:
# ---- Kaggle configuration ----
# Dataset is auto-discovered from /kaggle/input. If multiple matching datasets are attached,
# set RAW_ROOT manually to the correct folder after inspecting the printed candidates.
def find_parts_roots(input_root="/kaggle/input"):
    found = []
    for root, dirs, files in os.walk(input_root):
        if os.path.basename(root) == "train" and os.path.basename(os.path.dirname(root)) == "images":
            parent = os.path.dirname(os.path.dirname(root))
            if os.path.isdir(os.path.join(parent, "labels", "train")):
                if all(os.path.isdir(os.path.join(parent, "images", s)) and
                       os.path.isdir(os.path.join(parent, "labels", s))
                       for s in ("train", "val", "test")):
                    found.append(parent)
    return sorted(set(found))

PARTS_CANDIDATES = find_parts_roots()
print("Detected parts dataset candidates:")
for p in PARTS_CANDIDATES:
    print(" -", p)

RAW_ROOT = PARTS_CANDIDATES[0] if len(PARTS_CANDIDATES) == 1 else None
if RAW_ROOT is None:
    print("Set RAW_ROOT manually to the folder containing images/train, labels/train, images/val, labels/val, images/test and labels/test.")

# All outputs must be writable on Kaggle; /kaggle/input is read-only.
OUT_DRIVE_DIR = "/kaggle/working/autoinsight_models/parts"
OLD_WEIGHTS = None  # Optionally set to an attached previous best.pt path for comparison.

MODEL_CANDIDATES = ["yolo26m-seg.pt", "yolo11m-seg.pt"]
MODEL = MODEL_CANDIDATES[0]
IMGSZ = 800
EPOCHS = 50
TIME_H = 8.0        # Set within your Kaggle session quota; time cap may stop before EPOCHS.
BATCH = 4            # If CUDA OOM, lower to 4, then 2.
MERGE_SIDED = True
RUN_NAME = time.strftime("parts_seg_kaggle_%Y%m%d_%H%M")
PROMOTE = False      # Keep False until validation/test results and sample predictions are reviewed.

LOCAL_YOLO = "/kaggle/working/PARTS_YOLO"
YAML_PATH = f"{LOCAL_YOLO}/data.yaml"
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)

assert RAW_ROOT and os.path.isdir(os.path.join(RAW_ROOT, "images", "train")), (
    "Parts dataset root was not identified. Inspect /kaggle/input and set RAW_ROOT manually."
)
print("Dataset:", RAW_ROOT)
print("Output:", OUT_DRIVE_DIR)
print("Run:", RUN_NAME)


Detected parts dataset candidates:
 - /kaggle/input/datasets/devadeth888/carparts
Dataset: /kaggle/input/datasets/devadeth888/carparts
Output: /kaggle/working/autoinsight_models/parts
Run: parts_seg_kaggle_20261023_1859


## 1. Get carparts-seg (3,833 images, 23 classes)

In [7]:
# Validate the attached dataset. Unlike Colab, Kaggle input is already mounted read-only.
for s in ("train", "val", "test"):
    image_dir = f"{RAW_ROOT}/images/{s}"
    label_dir = f"{RAW_ROOT}/labels/{s}"
    assert os.path.isdir(image_dir), f"Missing image split: {image_dir}"
    assert os.path.isdir(label_dir), f"Missing label split: {label_dir}"
    images = [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]
    print(f"{s}: {len(images)} images, {len(labels)} label files")


train: 3156 images, 3156 label files
val: 401 images, 401 label files
test: 276 images, 276 label files


## 2. Remap classes (tested code)

In [8]:
# ---- carparts-seg: merge left/right variants, drop the catch-all 'object' class -----------
import os, shutil
from collections import Counter

ORIG_NAMES = ['back_bumper', 'back_door', 'back_glass', 'back_left_door', 'back_left_light', 'back_light',
              'back_right_door', 'back_right_light', 'front_bumper', 'front_door', 'front_glass',
              'front_left_door', 'front_left_light', 'front_light', 'front_right_door', 'front_right_light',
              'hood', 'left_mirror', 'object', 'right_mirror', 'tailgate', 'trunk', 'wheel']

# The sided classes have only 12-31 validation examples each, and the price table treats them
# identically to the unsided ones, so they are folded into one class each.
MERGE = {
    'back_left_door': 'back_door', 'back_right_door': 'back_door',
    'front_left_door': 'front_door', 'front_right_door': 'front_door',
    'back_left_light': 'back_light', 'back_right_light': 'back_light',
    'front_left_light': 'front_light', 'front_right_light': 'front_light',
    'left_mirror': 'mirror', 'right_mirror': 'mirror',
}
DROP = {'object'}      # catch-all class: it is not a repairable part


def build_class_map(orig_names, merge_sided=True):
    """old class id -> new class id (None = drop), plus the new class-name list."""
    new_names, mapping = [], {}
    for old_id, name in enumerate(orig_names):
        if name in DROP:
            mapping[old_id] = None
            continue
        target = MERGE.get(name, name) if merge_sided else name
        if target not in new_names:
            new_names.append(target)
        mapping[old_id] = new_names.index(target)
    return mapping, new_names


def remap_split(src_images, src_labels, dst_images, dst_labels, mapping):
    os.makedirs(dst_images, exist_ok=True)
    os.makedirs(dst_labels, exist_ok=True)
    stats, per_class = Counter(), Counter()
    for fname in os.listdir(src_images):
        shutil.copy2(os.path.join(src_images, fname), os.path.join(dst_images, fname))
    for fname in os.listdir(src_labels):
        if not fname.endswith('.txt'):
            continue
        out = []
        with open(os.path.join(src_labels, fname)) as f:
            for line in f.read().splitlines():
                parts = line.split()
                if not parts:
                    continue
                new = mapping.get(int(parts[0]))
                if new is None:
                    stats['dropped'] += 1
                    continue
                out.append(" ".join([str(new)] + parts[1:]))
                per_class[new] += 1
                stats['kept'] += 1
        with open(os.path.join(dst_labels, fname), 'w') as f:
            f.write("\n".join(out))
        stats['files'] += 1
    stats['_per_class'] = dict(per_class)
    return stats

In [9]:
mapping, new_names = build_class_map(ORIG_NAMES, MERGE_SIDED)
print(len(new_names), 'classes:', new_names)

shutil.rmtree(LOCAL_YOLO, ignore_errors=True)
for s in ('train', 'val', 'test'):
    st = remap_split(f'{RAW_ROOT}/images/{s}', f'{RAW_ROOT}/labels/{s}', f'{LOCAL_YOLO}/images/{s}', f'{LOCAL_YOLO}/labels/{s}', mapping)
    per = {new_names[k]: v for k, v in sorted(st.pop('_per_class').items())}
    print(s, dict(st)); print('   instances per class:', per)

with open(YAML_PATH, 'w') as f:
    yaml.safe_dump({'path': LOCAL_YOLO, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': new_names}, f, sort_keys=False)

# which classes would the repair-price table not know?
PRICE_TABLE_PARTS = {'front_bumper','back_bumper','front_door','back_door','front_left_door','front_right_door','back_left_door','back_right_door',
                     'front_glass','back_glass','front_light','back_light','front_left_light','front_right_light','back_left_light','back_right_light',
                     'wheel','hood','trunk','tailgate'}
print('classes with no repair price (will use a generic price, flagged in the UI):', [n for n in new_names if n not in PRICE_TABLE_PARTS])

13 classes: ['back_bumper', 'back_door', 'back_glass', 'back_light', 'front_bumper', 'front_door', 'front_glass', 'front_light', 'hood', 'mirror', 'tailgate', 'trunk', 'wheel']
train {'kept': 16723, 'files': 3156, 'dropped': 10}
   instances per class: {'back_bumper': 732, 'back_door': 1396, 'back_glass': 890, 'back_light': 2036, 'front_bumper': 1616, 'front_door': 1582, 'front_glass': 1682, 'front_light': 3324, 'hood': 1706, 'mirror': 816, 'tailgate': 88, 'trunk': 146, 'wheel': 709}
val {'kept': 2041, 'files': 401, 'dropped': 1}
   instances per class: {'back_bumper': 94, 'back_door': 186, 'back_glass': 115, 'back_light': 258, 'front_bumper': 208, 'front_door': 194, 'front_glass': 214, 'front_light': 429, 'hood': 214, 'mirror': 62, 'tailgate': 5, 'trunk': 9, 'wheel': 53}
test {'dropped': 1, 'files': 276, 'kept': 1598}
   instances per class: {'back_bumper': 44, 'back_door': 149, 'back_glass': 53, 'back_light': 140, 'front_bumper': 154, 'front_door': 160, 'front_glass': 166, 'front_lig

### Add your own photos (the biggest accuracy lever)
Label service-centre photos of **side views and close-ups of doors** (the cases that fail today) with the **same class names** as above, export in YOLO-segmentation format, and copy them into
`/content/PARTS_YOLO/images/train` and `/content/PARTS_YOLO/labels/train` (and ~15% into `val`) before training. Keep whole vehicles out of both splits at once.

In [10]:
def evaluate(weights, yaml_path, split='val', imgsz=640, conf=None):
    """Box (and mask, for -seg models) metrics, overall and per class."""
    m = YOLO(weights)
    kw = dict(data=yaml_path, split=split, imgsz=imgsz, plots=False, verbose=False)
    if conf is not None:
        kw['conf'] = conf                       # conf=0.35 = the operating point the server uses
    r = m.val(**kw)
    out = {'weights': weights, 'split': split, 'conf': conf, 'imgsz': imgsz,
           'box_mAP50': float(r.box.map50), 'box_mAP50_95': float(r.box.map),
           'precision': float(r.box.mp), 'recall': float(r.box.mr), 'per_class': {}}
    seg = getattr(r, 'seg', None)
    if seg is not None and hasattr(seg, 'map'):
        out['mask_mAP50'] = float(seg.map50)
        out['mask_mAP50_95'] = float(seg.map)
    for j, c in enumerate(r.box.ap_class_index):
        p, rec, ap50, ap = r.box.class_result(j)
        out['per_class'][m.names[int(c)]] = {'P': round(float(p), 3), 'R': round(float(rec), 3),
                                              'mAP50': round(float(ap50), 3), 'mAP50-95': round(float(ap), 3)}
    return out


def show(res, title=''):
    print(f"\n== {title} [{res['split']}, conf={res['conf']}, imgsz={res['imgsz']}]")
    keys = ['box_mAP50', 'box_mAP50_95', 'mask_mAP50', 'mask_mAP50_95', 'precision', 'recall']
    print('   ' + '  '.join(f"{k}={res[k]:.3f}" for k in keys if k in res))
    print(f"   {'class':16} {'P':>6} {'R':>6} {'mAP50':>7} {'mAP50-95':>9}")
    for name, v in res['per_class'].items():
        print(f"   {name:16} {v['P']:>6} {v['R']:>6} {v['mAP50']:>7} {v['mAP50-95']:>9}")

In [11]:
if OLD_WEIGHTS and os.path.isfile(OLD_WEIGHTS):
    print("Optional old weights supplied:", OLD_WEIGHTS)
    old_test = evaluate(OLD_WEIGHTS, YAML_PATH, "test", imgsz=IMGSZ)
    show(old_test, "OLD weights on the remapped test dataset")
else:
    old_test = None


## 3. Train

In [14]:
# Try the medium model first for higher capacity. Fall back only if the checkpoint cannot load.
model = None
load_errors = []
for candidate in MODEL_CANDIDATES:
    try:
        model = YOLO(candidate)
        MODEL = candidate
        print("Using pretrained segmentation model:", MODEL)
        break
    except Exception as e:
        load_errors.append(f"{candidate}: {e}")
if model is None:
    raise RuntimeError("Could not load any segmentation checkpoint:\n" + "\n".join(load_errors))

results = model.train(
    data=YAML_PATH,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    time=TIME_H,
    patience=40,
    batch=BATCH,
    device=[0,1],
    cos_lr=True,
    close_mosaic=15,
    degrees=5,
    scale=0.6,
    cache=False,
    workers=4,
    seed=42,
    deterministic=True,
    project=f"{OUT_DRIVE_DIR}/runs",
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
)


Using pretrained segmentation model: yolo26m-seg.pt
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/PARTS_YOLO/data.yaml, degrees=5, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      2/144      3.62G      1.165      1.879       1.81    0.01271      2.197         41        800: 100% ━━━━━━━━━━━━ 789/789 4.4it/s 3:010.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 6.7it/s 7.6s0.1s
                   all        401       2041      0.632      0.646      0.655       0.44      0.626       0.66      0.649       0.45

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      3/148      3.62G      1.191       1.83      1.598    0.01283      1.726         21        800: 100% ━━━━━━━━━━━━ 789/789 4.4it/s 3:010.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 6.7it/s 7.6s0.2s
                   all   

In [16]:
import os

paths = [
    "/kaggle/working/yolo26m-seg.pt",
]

for root, dirs, files in os.walk("/kaggle/working"):
    for filename in files:
        if filename in ["best.pt", "last.pt", "yolo26m-seg.pt"]:
            paths.append(os.path.join(root, filename))

for path in sorted(set(paths)):
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{path}: {size_mb:.2f} MB")

/kaggle/working/autoinsight_models/parts/runs/parts_seg_kaggle_20261023_1859/weights/best.pt: 52.04 MB
/kaggle/working/autoinsight_models/parts/runs/parts_seg_kaggle_20261023_1859/weights/last.pt: 52.04 MB
/kaggle/working/yolo26m-seg.pt: 52.21 MB


In [15]:
from pathlib import Path
from ultralytics import YOLO

last_path = Path(
    "/kaggle/working/autoinsight_models/parts/runs/"
    "YOUR_RUN_FOLDER/weights/last.pt"
)

assert last_path.exists(), f"Checkpoint not found: {last_path}"

model = YOLO(str(last_path))
model.train(resume=True)

AssertionError: Checkpoint not found: /kaggle/working/autoinsight_models/parts/runs/YOUR_RUN_FOLDER/weights/last.pt

## Resume after a Kaggle interruption

Kaggle may remove `/kaggle/working` when a session ends. Save a notebook version with outputs or download/persist the run folder first. If the run folder is still present, run the setup and dataset-preparation cells, then:

```python
last_path = f"{OUT_DRIVE_DIR}/runs/{RUN_NAME}/weights/last.pt"
model = YOLO(last_path)
results = model.train(resume=True)
```

For a new session, restore the previous run folder from a Kaggle Dataset or upload it as notebook input, then update `OUT_DRIVE_DIR`/`RUN_NAME` accordingly.


## 4. Evaluate on validation AND test

In [ ]:
RUN_DIR = f"{OUT_DRIVE_DIR}/runs/{RUN_NAME}"
NEW_BEST = f"{RUN_DIR}/weights/best.pt"
assert os.path.isfile(NEW_BEST), f"Best checkpoint not found: {NEW_BEST}"

new = {
    "val": evaluate(NEW_BEST, YAML_PATH, "val", imgsz=IMGSZ),
    "test": evaluate(NEW_BEST, YAML_PATH, "test", imgsz=IMGSZ),
}
show(new["val"], "NEW — validation")
show(new["test"], "NEW — final held-out test")
if old_test:
    print("Old/new comparisons are meaningful only when class definitions and test data match.")


## 5. Save a model card, and (optionally) promote

In [ ]:
card = {
    "run": RUN_NAME,
    "date": time.strftime("%Y-%m-%d %H:%M"),
    "base_model": MODEL,
    "imgsz": IMGSZ,
    "epochs_planned": EPOCHS,
    "time_cap_h": TIME_H,
    "ultralytics": ultralytics.__version__,
    "classes": new_names,
    "task": "segment",
    "dataset": {s: len(os.listdir(f"{LOCAL_YOLO}/images/{s}")) for s in ("train", "val", "test")},
    "metrics": new,
}
with open(f"{RUN_DIR}/model_card.json", "w") as f:
    json.dump(card, f, indent=2)

# Export to /kaggle/working for download or "Save Version" output.
EXPORT_DIR = "/kaggle/working/parts_model_export"
os.makedirs(EXPORT_DIR, exist_ok=True)
shutil.copy2(NEW_BEST, f"{EXPORT_DIR}/best.pt")
with open(f"{EXPORT_DIR}/classes.json", "w") as f:
    json.dump(new_names, f, indent=2)
shutil.copy2(f"{RUN_DIR}/model_card.json", f"{EXPORT_DIR}/model_card.json")
shutil.make_archive("/kaggle/working/parts_model_export", "zip", EXPORT_DIR)
print("Export created: /kaggle/working/parts_model_export.zip")

if PROMOTE:
    target = f"{OUT_DRIVE_DIR}/best.pt"
    if os.path.exists(target):
        backup = f"{OUT_DRIVE_DIR}/best_prev_{time.strftime('%Y%m%d_%H%M')}.pt"
        shutil.copy2(target, backup)
        print("Backed up previous checkpoint:", backup)
    shutil.copy2(NEW_BEST, target)
    print("Promoted:", target)
else:
    print("Not promoted. Review metrics and predictions before changing PROMOTE=True.")
print("Backend target after approval: backend/models/parts/best.pt")


## Accuracy checklist before accepting this model

1. Inspect `results.png`, confusion matrix and validation prediction plots.
2. Review per-class mask mAP and recall, especially doors, lights, mirrors and wheels.
3. Run inference on your own vehicle images and service-centre photos.
4. Do not tune thresholds against the test set. Tune on validation data, then evaluate test once.
5. A larger model may improve accuracy but can overfit or run slower. Compare YOLO26m-seg and YOLO26s-seg on the same split if GPU time allows.
6. Your backend class names must match `classes.json`. Merging left/right labels changes the output schema.
